# 04 — Width scaling rules

This notebook confirms how the benchmark rescales each architecture's capacity. Every architecture has one or more width attributes that the pipeline multiplies by a scale factor (snapping to a divisor) so that parameter count can be tuned smoothly. We visualise the monotone relationship between scale and parameter count, which is the property the size-matcher relies on.

Modules exercised: `pipelines/benchmark_pipeline/sizing.py` (`WidthScaler`, `WidthRule`, `WidthScaler.overrides`, `WidthScaler.scaled_config`).

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../..").resolve()))

import warnings
warnings.filterwarnings("ignore")

import numpy             as np
import torch
import matplotlib.pyplot as plt

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(False)

DEVICE = torch.device("cpu")

plt.rcParams.update({
    "figure.dpi"     : 110,
    "font.size"      : 10,
    "axes.grid"      : True,
    "grid.alpha"     : 0.3,
    "axes.axisbelow" : True,
})

print("bootstrap ready  —  torch", torch.__version__, " device", DEVICE)


## The per-architecture width rules

`WidthScaler` holds, for each architecture key, a list of `WidthRule` entries naming the config attribute to scale and the divisor it snaps to. We print the rule table so the scaling targets are explicit.

In [ ]:
from pipelines.benchmark_pipeline.sizing import WidthScaler

scaler = WidthScaler()

for name, rules in scaler.rules.items():
    descr = ", ".join(f"{r.attribute}/÷{r.divisor}" for r in rules)
    print(f"{name:22s} {descr}")

## Overrides produced at a few scales

For one representative architecture we show the concrete attribute overrides produced at several scales. The values move monotonically and stay snapped to the rule divisor.

In [ ]:
demo = "unet"
for scale in (0.1, 0.25, 0.5, 1.0, 2.0):
    print(f"scale {scale:>4}: {scaler.overrides(demo, scale)}")

## Parameter count versus scale

We sweep the scale factor for several architectures and count the parameters of the resulting scaled configuration. The curves should rise monotonically, which is what lets the bisection in the next notebook converge on a target budget.

In [ ]:
from models import get_model
from pipelines.benchmark_pipeline.sizing import _IMAGE_SIZE_MODELS

IN_CHANNELS  = 9
OUT_CHANNELS = 15
IMAGE_SIZE   = 64

def count_at(name, scale):
    config    = scaler.scaled_config(name, scale)
    overrides = {"in_channels": IN_CHANNELS, "out_channels": OUT_CHANNELS}
    if name in _IMAGE_SIZE_MODELS:
        overrides["image_size"] = IMAGE_SIZE
    model, _ = get_model(name, config=config, **overrides)
    n        = sum(p.numel() for p in model.parameters())
    del model
    return n

scales  = np.linspace(0.1, 1.5, 10)
curves  = {}
for name in ["unet", "resunet", "linknet", "dense_unet"]:
    curves[name] = [count_at(name, float(s)) for s in scales]
    print(f"{name:12s} {curves[name][0]:>8,} ... {curves[name][-1]:>8,}")

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 5))
for name, ys in curves.items():
    ax.plot(scales, np.array(ys) / 1e6, marker="o", label=name)

ax.set_xlabel("width scale factor")
ax.set_ylabel("parameters (millions)")
ax.set_title("Parameter count is monotone in the width scale")
ax.legend()
fig.tight_layout()
plt.show()

## Expected visual outcome

The rule table lists a scaling target for all twenty-one architectures, the override demonstration shows values snapped to the divisor and increasing with scale, and every curve in the plot is non-decreasing. The monotonicity is the precondition the size-matching bisection depends on.